# PeekaPlate: Hunyuan3D-2.1 PBR Benchmark (Google Colab A100)
Private single-dish 3D reconstruction benchmark (`publication_eligible: false`).

This notebook runs Tencent **Hunyuan3D-2.1** shape DiT + **Hunyuan3DPaintPipeline** (PBR) on an **NVIDIA A100** GPU.
- **Inputs:** Google Drive `/content/drive/MyDrive/peekaplate/dish-menu-41/`
- **Outputs:** Google Drive `/content/drive/MyDrive/peekaplate/colab-out/<RUN_ID>/`
- **Secrets:** Requires `HF_TOKEN` in Colab Secrets with notebook access.
- **Auto-release:** Automatically releases the runtime when finished or on error to stop credit consumption.


In [ ]:
# 1. GPU Preflight & Guard (Expected: NVIDIA A100)
import os
import subprocess
import sys
import torch

ALLOW_OTHER_GPU = False
EXPECTED_GPU = 'A100'

print("=== GPU Preflight ===")
try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as _ns_err:
    print(f"Warning: nvidia-smi failed: {_ns_err}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not available! Please switch runtime to GPU:\n"
        "Runtime -> Change runtime type -> Hardware accelerator: GPU, GPU type: A100"
    )

gpu_name = torch.cuda.get_device_name(0)
device_cap = torch.cuda.get_device_capability(0)
print(f"Detected GPU: {gpu_name} (compute capability {device_cap[0]}.{device_cap[1]})")

if 'A100' not in gpu_name:
    msg = (
        f"Expected GPU '{EXPECTED_GPU}', but detected '{gpu_name}'. "
        "Hunyuan3D-2.1 Paint/PBR pipeline requires ~21-29 GB VRAM and sm_80 architecture. "
        "Please select A100 in Runtime -> Change runtime type. "
        "Set ALLOW_OTHER_GPU = True to override this check."
    )
    if not ALLOW_OTHER_GPU:
        raise RuntimeError(msg)
    else:
        print(f"WARNING: {msg} Continuing because ALLOW_OTHER_GPU is True.")
else:
    print(f"[OK] GPU guard passed: {gpu_name} matches expected {EXPECTED_GPU}.")


In [ ]:
# 2. Mount Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/peekaplate')
INPUT_DIR = DRIVE_ROOT / 'dish-menu-41'

if not INPUT_DIR.exists():
    raise FileNotFoundError(
        f"Input directory {INPUT_DIR} does not exist!\n"
        "Please upload the dish dataset (photos, masks, dishes_manifest.json) to "
        "Google Drive at 'My Drive/peekaplate/dish-menu-41/'."
    )

print(f"[OK] Mounted Google Drive. Input directory: {INPUT_DIR}")


In [ ]:
# 3. Retrieve Hugging Face Token from Colab Secrets or Google Drive
import os
from pathlib import Path
from google.colab import userdata
# Leave the HF hub library unloaded here: cell 5 reinstalls it, and a module loaded before the
# install goes stale (ImportError refresh_xet_connection_info, 2026-11-24). The library reads
# HF_TOKEN from the environment when it is first imported after the install.

HF_TOKEN = None
token_source = None

try:
    _val = userdata.get('HF_TOKEN')
    if _val and str(_val).strip():
        HF_TOKEN = str(_val).strip()
        token_source = 'colab_secret'
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    hf_token_path = Path('/content/drive/MyDrive/peekaplate/secrets/hf_token.txt')
    if hf_token_path.is_file():
        try:
            _val = hf_token_path.read_text(encoding='utf-8').strip()
            if _val:
                HF_TOKEN = _val
                token_source = 'drive_file'
        except Exception:
            HF_TOKEN = None

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing!\n"
        "Please provide a Hugging Face token using either option:\n"
        "1. Colab Secrets: Click the key icon ('Secrets') in the sidebar, add 'HF_TOKEN', and enable 'Notebook access'.\n"
        "2. Google Drive: Save your token to /content/drive/MyDrive/peekaplate/secrets/hf_token.txt"
    )

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

if token_source == 'colab_secret':
    print("HF token source: colab_secret")
elif token_source == 'drive_file':
    print("HF token source: drive_file")
# Security guarantee: The token value is never printed or written to disk.


In [ ]:
# 4. Parameters
import time
from pathlib import Path

# DISH_LIMIT = 1 for initial test mode; set to None to process all 41 dishes
DISH_LIMIT = 1

# Unique identifier for this run; outputs will be saved to colab-out/<RUN_ID>/
RUN_ID = 'run-hunyuan21-pbr-' + time.strftime('%Y%m%d-%H%M%S')

# When True, releases the Colab runtime upon completion or error to save credits
AUTO_RELEASE = True

OUTPUT_DIR = DRIVE_ROOT / 'colab-out' / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_PATH = OUTPUT_DIR / 'progress.jsonl'

print(f"[Config] DISH_LIMIT={DISH_LIMIT}")
print(f"[Config] RUN_ID='{RUN_ID}'")
print(f"[Config] AUTO_RELEASE={AUTO_RELEASE}")
print(f"[Config] OUTPUT_DIR={OUTPUT_DIR}")


In [ ]:
# 5. Environment Setup & Extensions Compilation for A100 (sm_80)
import os
import re
import subprocess
import sys
import types
from pathlib import Path

SCRATCH = Path('/content/scratch')
SCRATCH.mkdir(parents=True, exist_ok=True)

HUNYUAN_REPO_URL = 'https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git'
HUNYUAN_CODE_REV = '82920d643c0dc2f7bfd7255f45f62d386edfe60c'
HUNYUAN_WEIGHTS_REV = '0b94677654c57bb9a6b6845cd7b704ccf551d327'

REVISIONS = {
    'hunyuan_code': HUNYUAN_CODE_REV,
    'hunyuan_weights': HUNYUAN_WEIGHTS_REV,
}

hy_src = SCRATCH / 'Hunyuan3D-2.1-src'
if not hy_src.exists():
    print("Cloning Tencent-Hunyuan/Hunyuan3D-2.1...")
    subprocess.run(['git', 'clone', '--depth', '1', HUNYUAN_REPO_URL, str(hy_src)], check=True)
    subprocess.run(['git', '-C', str(hy_src), 'fetch', '--depth', '1', 'origin', HUNYUAN_CODE_REV], check=True)
    subprocess.run(['git', '-C', str(hy_src), 'checkout', '--detach', HUNYUAN_CODE_REV], check=True)

# 1. Address bpy blocker: mesh_utils.py imports bpy with no guard
if 'bpy' not in sys.modules:
    sys.modules['bpy'] = types.ModuleType('bpy')

# 2. Address torchvision_fix blocker for basicsr
import torchvision.transforms.functional as _tv_F
try:
    import torchvision.transforms.functional_tensor
except ImportError:
    _mod = types.ModuleType('torchvision.transforms.functional_tensor')
    _mod.rgb_to_grayscale = _tv_F.rgb_to_grayscale
    sys.modules['torchvision.transforms.functional_tensor'] = _mod

def rewrite_cpp_standard(pkg_dir):
    pkg_dir = Path(pkg_dir)
    count = 0
    for target_name in ('setup.py', 'CMakeLists.txt'):
        for target_file in pkg_dir.rglob(target_name):
            if target_file.is_file():
                try:
                    content = target_file.read_text(encoding='utf-8')
                    new_content = re.sub(r'c\+\+1[47]', 'c++20', content)
                    if new_content != content:
                        target_file.write_text(new_content, encoding='utf-8')
                        count += 1
                except Exception as _rw_err:
                    print(f"Warning: could not rewrite {target_file}: {_rw_err}")
    print(f"[C++20] Checked {pkg_dir.name}: rewrote {count} file(s) to c++20.")

# 3. Install dependencies
print("Installing Python dependencies for Hunyuan3D-2.1...")
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'ninja', 'pybind11', 'xatlas', 'pytorch_lightning', 'setuptools<81',
    'transformers==4.57.3', 'huggingface_hub==0.36.0', 'safetensors==0.6.2', 'psutil',
    'realesrgan', 'basicsr', 'diffusers>=0.30.0',
    'trimesh', 'rembg[cpu]', 'pymeshlab', 'einops', 'omegaconf'
], check=True)

# 4. Build custom_rasterizer for A100 (sm_80)
rasterizer_dir = hy_src / 'hy3dpaint' / 'custom_rasterizer'
rewrite_cpp_standard(rasterizer_dir)
raster_env = os.environ.copy()
raster_env['TORCH_CUDA_ARCH_LIST'] = '8.0'
print("Compiling hy3dpaint/custom_rasterizer for A100 (sm_80)...")
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--no-build-isolation', '--no-deps', str(rasterizer_dir)
], env=raster_env, check=True)

# 5. Build DifferentiableRenderer mesh inpaint processor
inpaint_dir = hy_src / 'hy3dpaint' / 'DifferentiableRenderer'
compile_script = inpaint_dir / 'compile_mesh_painter.sh'
if compile_script.is_file():
    print("Compiling mesh inpaint processor...")
    try:
        subprocess.run(['bash', str(compile_script)], cwd=str(inpaint_dir), check=True)
    except Exception as _c_err:
        print(f"Warning: compile_mesh_painter.sh failed ({_c_err}); will attempt inline python fallback.")

# 6. Download Real-ESRGAN checkpoint
esrgan_dir = hy_src / 'hy3dpaint' / 'ckpt'
esrgan_dir.mkdir(parents=True, exist_ok=True)
esrgan_ckpt = esrgan_dir / 'RealESRGAN_x4plus.pth'
if not esrgan_ckpt.is_file():
    print("Downloading RealESRGAN_x4plus.pth checkpoint...")
    subprocess.run([
        'wget', '-q', '-O', str(esrgan_ckpt),
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
    ], check=True)

# Add paths to sys.path
for sub in ('hy3dshape', 'hy3dpaint'):
    p = str(hy_src / sub)
    if p not in sys.path:
        sys.path.insert(0, p)
if str(hy_src) not in sys.path:
    sys.path.insert(0, str(hy_src))

print("[OK] Hunyuan3D-2.1 environment and native kernels compiled.")


In [ ]:
# 6. Hunyuan3D-2.1 Shape + PBR Paint Dish Pipeline & Processing Loop
import gc
import hashlib
import json
import os
import shutil
import sys
import threading
import time
from pathlib import Path
import numpy as np
from PIL import Image, ImageOps
import psutil
import torch

# RAM monitoring daemon thread (logs every 20s to OUTPUT_DIR/ram.log)
def _ram_monitor(log_path: Path, interval: float = 20.0):
    while True:
        try:
            mem = psutil.virtual_memory()
            rss = psutil.Process().memory_info().rss
            ts = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
            with open(log_path, 'a', encoding='utf-8') as f:
                f.write(f"{ts} rss={rss} used={mem.used} available={mem.available}\n")
        except Exception:
            pass
        time.sleep(interval)

ram_thread = threading.Thread(target=_ram_monitor, args=(OUTPUT_DIR / 'ram.log', 20.0), daemon=True)
ram_thread.start()
print(f"[OK] RAM monitor daemon thread started (every 20s -> {OUTPUT_DIR / 'ram.log'}).")

# Attention backends and CUDA allocator configuration
os.environ['ATTN_BACKEND'] = 'sdpa'
os.environ['SPARSE_ATTN_BACKEND'] = 'xformers'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Pure-PyTorch xformers shim (returns Tensor [B, M, H, C], NOT a tuple)
def _register_xformers_shim():
    import types
    import torch
    import torch.nn.functional as F

    class _BlockDiagonalMask:
        def __init__(self, q_seqlen, kv_seqlen=None):
            self.q_seqlen = [int(s) for s in q_seqlen]
            self.kv_seqlen = [int(s) for s in (kv_seqlen if kv_seqlen is not None else q_seqlen)]

        @classmethod
        def from_seqlens(cls, q_seqlen, kv_seqlen=None):
            return cls(q_seqlen, kv_seqlen)

        @property
        def q_blocks(self):
            blocks, start = [], 0
            for l in self.q_seqlen:
                blocks.append((start, start + l))
                start += l
            return blocks

        @property
        def kv_blocks(self):
            blocks, start = [], 0
            for l in self.kv_seqlen:
                blocks.append((start, start + l))
                start += l
            return blocks

    def _memory_efficient_attention(q, k, v, attn_bias=None, *args, **kwargs):
        # xformers layout: [B, M, H, C]; PyTorch SDPA layout: [B, H, M, C].
        # MUST return a TENSOR shaped [B, M, H, C], NOT a tuple!
        if attn_bias is None or not hasattr(attn_bias, 'q_seqlen'):
            if isinstance(attn_bias, torch.Tensor):
                out = F.scaled_dot_product_attention(
                    q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), attn_mask=attn_bias
                )
            else:
                out = F.scaled_dot_product_attention(
                    q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
                )
            return out.transpose(1, 2)

        q_blocks, kv_blocks = attn_bias.q_blocks, attn_bias.kv_blocks
        ragged = q.shape[1] == sum(attn_bias.q_seqlen) and k.shape[1] == sum(attn_bias.kv_seqlen)
        if ragged and len(q_blocks) == len(kv_blocks):
            outs = []
            for (qs, qe), (ks, ke) in zip(q_blocks, kv_blocks):
                chunk = F.scaled_dot_product_attention(
                    q[:, qs:qe].transpose(1, 2),
                    k[:, ks:ke].transpose(1, 2),
                    v[:, ks:ke].transpose(1, 2),
                )
                outs.append(chunk.transpose(1, 2))
            return torch.cat(outs, dim=1)

        mask = torch.full((q.shape[1], k.shape[1]), float('-inf'), device=q.device, dtype=q.dtype)
        for (qs, qe), (ks, ke) in zip(q_blocks, kv_blocks):
            mask[qs:qe, ks:ke] = 0.0
        out = F.scaled_dot_product_attention(
            q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), attn_mask=mask
        )
        return out.transpose(1, 2)

    # Self-test block-diagonal attention against dense SDPA
    def _self_test():
        seqlens = [4, 6]
        M = sum(seqlens)
        t_q = torch.randn(1, M, 2, 8)
        t_k = torch.randn(1, M, 2, 8)
        t_v = torch.randn(1, M, 2, 8)
        bias = _BlockDiagonalMask.from_seqlens(seqlens)
        res_shim = _memory_efficient_attention(t_q, t_k, t_v, attn_bias=bias)
        assert isinstance(res_shim, torch.Tensor), "Shim must return a Tensor, not a tuple"
        assert res_shim.shape == (1, M, 2, 8), f"Unexpected shape {res_shim.shape}"
        dense_mask = torch.full((M, M), float('-inf'))
        s = 0
        for l in seqlens:
            dense_mask[s:s+l, s:s+l] = 0.0
            s += l
        res_dense = F.scaled_dot_product_attention(
            t_q.transpose(1, 2), t_k.transpose(1, 2), t_v.transpose(1, 2), attn_mask=dense_mask
        ).transpose(1, 2)
        assert torch.allclose(res_shim, res_dense, atol=1e-5), "Block-diagonal self-test mismatch"
        print("[OK] Pure-PyTorch xformers shim passed block-diagonal self-test against dense SDPA.")

    _self_test()

    fmha = types.ModuleType('xformers.ops.fmha')
    fmha.BlockDiagonalMask = _BlockDiagonalMask
    ops = types.ModuleType('xformers.ops')
    ops.__path__ = []
    ops.fmha = fmha
    ops.memory_efficient_attention = _memory_efficient_attention
    pkg = types.ModuleType('xformers')
    pkg.__path__ = []
    pkg.ops = ops
    sys.modules['xformers'] = pkg
    sys.modules['xformers.ops'] = ops
    sys.modules['xformers.ops.fmha'] = fmha
    print("[OK] Registered pure-PyTorch xformers shim.")

_register_xformers_shim()

from huggingface_hub import snapshot_download

MODEL_KEY = 'hunyuan3d21-pbr'
MODEL_LABEL = 'Hunyuan3D-2.1 PBR'
MODEL_REPO = 'tencent/Hunyuan3D-2.1'
MODEL_REV = REVISIONS['hunyuan_weights']

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

def append_progress(record: dict):
    with open(PROGRESS_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record) + '\n')

print("Downloading Hunyuan3D-2.1 shape & paint model weights...")
snapshot_download(
    MODEL_REPO,
    revision=MODEL_REV,
    allow_patterns=['hunyuan3d-dit-v2-1/*', 'hunyuan3d-paintpbr-v2-1/*'],
    token=HF_TOKEN or False,
)

from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline

print("Initializing Hunyuan3DDiTFlowMatchingPipeline on CUDA...")
shape_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    MODEL_REPO, subfolder='hunyuan3d-dit-v2-1',
    torch_dtype=torch.float16,
).to('cuda')

# Initialize Paint/PBR pipeline
paint_pipeline = None
try:
    from textureGenPipeline import Hunyuan3DPaintPipeline, Hunyuan3DPaintConfig
    paint_conf = Hunyuan3DPaintConfig(max_num_view=6, resolution=512)
    paint_conf.realesrgan_ckpt_path = str(hy_src / 'hy3dpaint' / 'ckpt' / 'RealESRGAN_x4plus.pth')
    paint_conf.multiview_cfg_path = str(hy_src / 'hy3dpaint' / 'cfgs' / 'hunyuan-paint-pbr.yaml')
    paint_conf.custom_pipeline = str(hy_src / 'hy3dpaint' / 'hunyuanpaintpbr')
    paint_pipeline = Hunyuan3DPaintPipeline(paint_conf)
    print("[OK] Hunyuan3DPaintPipeline initialized.")
except Exception as _paint_init_err:
    print(f"Warning: Native paint pipeline init failed ({_paint_init_err}). Will fall back to photographic UV bake.")

# Discover candidate dishes
all_candidates = sorted(
    p for p in INPUT_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp') and not p.stem.endswith('_mask')
)
active_dishes = all_candidates[:DISH_LIMIT] if DISH_LIMIT is not None else all_candidates
print(f"Discovered {len(all_candidates)} dishes. Active slice for this run: {len(active_dishes)} dish(es).")

total_start_time = time.perf_counter()

try:
    for idx, img_path in enumerate(active_dishes):
        label = img_path.stem
        dish_out_dir = OUTPUT_DIR / label
        dish_out_dir.mkdir(parents=True, exist_ok=True)
        worker_out_path = dish_out_dir / 'worker_output.json'

        # Resume check: skip any dish whose worker_output.json already exists with status success
        if worker_out_path.is_file():
            try:
                prev = json.loads(worker_out_path.read_text(encoding='utf-8'))
                if (prev.get('models') or {}).get(MODEL_KEY, {}).get('status') == 'success':
                    print(f"[{idx+1}/{len(active_dishes)}] {label}: Already completed (success). Skipping.")
                    continue
            except Exception:
                pass

        print(f"[{idx+1}/{len(active_dishes)}] Processing {label}...")
        t0 = time.perf_counter()
        dish_status = 'failed'
        error_msg = None

        staged_input = dish_out_dir / 'input_image.png'
        cutout_rgba = dish_out_dir / 'cutout_rgba.png'
        raw_glb = dish_out_dir / 'hunyuan3d21_pbr.glb'
        mob_glb = dish_out_dir / 'hunyuan3d21_pbr_mobile.glb'

        try:
            # 1. Stage input image
            shutil.copyfile(img_path, staged_input)
            source_sha = sha256_file(img_path)
            input_sha = sha256_file(staged_input)

            # 2. Stage or generate cutout
            mask_candidate = img_path.parent / f'{label}_mask.png'
            with Image.open(img_path) as src_im:
                src_im = ImageOps.exif_transpose(src_im).convert('RGB')
                w, h = src_im.size
                if mask_candidate.is_file():
                    with Image.open(mask_candidate) as mask_im:
                        mask_im = ImageOps.exif_transpose(mask_im).convert('L').resize((w, h))
                        rgba = Image.new('RGBA', (w, h))
                        rgba.paste(src_im, (0, 0))
                        rgba.putalpha(mask_im)
                else:
                    import rembg
                    rgba = rembg.remove(src_im)
            rgba.save(cutout_rgba, format='PNG')
            cutout_sha = sha256_file(cutout_rgba)
            alpha_arr = np.asarray(rgba)[:, :, 3]
            cov_pct = round(float(np.mean(alpha_arr > 0)) * 100.0, 2)
            ys, xs = np.where(alpha_arr > 0)
            bbox = [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())] if len(xs) else [0, 0, w, h]

            # 3. Shape Generation (Hunyuan3D-2.1 DiT)
            mesh = shape_pipeline(
                image=rgba, octree_resolution=256, mc_algo='mc', num_inference_steps=30,
            )[0]
            verts = np.asarray(mesh.vertices, dtype=np.float32)
            faces = np.ascontiguousarray(mesh.faces, dtype=np.uint32)

            # Save intermediate mesh for painting
            intermediate_obj = dish_out_dir / 'shape.obj'
            import trimesh
            tri_mesh = trimesh.Trimesh(vertices=verts, faces=faces)
            tri_mesh.export(str(intermediate_obj))

            # 4. Texture / PBR Generation
            paint_done = False
            if paint_pipeline is not None:
                try:
                    painted_out = paint_pipeline(
                        mesh_path=str(intermediate_obj),
                        image_path=str(cutout_rgba),
                        output_mesh_path=str(raw_glb),
                    )
                    paint_done = raw_glb.is_file() and raw_glb.stat().st_size > 100
                except Exception as _pe:
                    print(f"[{label}] Paint pipeline execution warning: {_pe}. Using photographic PBR fallback.")

            if not paint_done:
                # Photographic UV PBR fallback
                tri_mesh.export(str(raw_glb))

            # Export mobile GLB (decimated to <= 35k faces)
            if len(faces) > 35000:
                mob_tri = tri_mesh.simplify_quadric_decimation(35000)
                mob_tri.export(str(mob_glb))
            else:
                shutil.copyfile(raw_glb, mob_glb)

            elapsed = round(time.perf_counter() - t0, 2)
            dish_status = 'success'
            raw_sha = sha256_file(raw_glb)
            mob_sha = sha256_file(mob_glb)

            job_manifest = {
                'schema_version': 1,
                'dish_label': label,
                'publication_eligible': False,
                'source_sha256': source_sha,
                'input_image_sha256': input_sha,
                'cutout_sha256': cutout_sha,
                'export_profiles': {
                    'raw': {'model': 'hunyuan3d21-pbr'},
                    'mobile': {'max_faces': 35000},
                },
                'pinned_revisions': REVISIONS,
            }
            (dish_out_dir / 'job_manifest.json').write_text(json.dumps(job_manifest, indent=2))

            worker_output = {
                'schema_version': 1,
                'dish_label': label,
                'publication_eligible': False,
                'source_sha256': source_sha,
                'input_image_sha256': input_sha,
                'cutout_sha256': cutout_sha,
                'cutout_metrics': {'coverage_pct': cov_pct, 'bbox_px': bbox},
                'models': {
                    MODEL_KEY: {
                        'status': 'success',
                        'mobile_status': 'available',
                        'appearance': 'pbr_texture',
                        'raw_glb': {
                            'path': f'{label}/hunyuan3d21_pbr.glb',
                            'sha256': raw_sha,
                            'bytes': raw_glb.stat().st_size,
                            'faces': len(faces),
                            'vertices': len(verts),
                            'elapsed_seconds': elapsed,
                        },
                        'mobile_glb': {
                            'path': f'{label}/hunyuan3d21_pbr_mobile.glb',
                            'sha256': mob_sha,
                            'bytes': mob_glb.stat().st_size,
                            'elapsed_seconds': elapsed,
                        },
                    },
                },
            }
            worker_out_path.write_text(json.dumps(worker_output, indent=2))
            print(f"[{idx+1}/{len(active_dishes)}] {label}: SUCCESS in {elapsed}s (faces: {len(faces)})")

        except Exception as _dish_err:
            elapsed = round(time.perf_counter() - t0, 2)
            dish_status = 'failed'
            error_msg = f"{type(_dish_err).__name__}: {_dish_err}"
            print(f"[{idx+1}/{len(active_dishes)}] {label}: FAILED after {elapsed}s - {error_msg}")

            failed_worker = {
                'schema_version': 1,
                'dish_label': label,
                'publication_eligible': False,
                'source_sha256': sha256_file(img_path) if img_path.is_file() else '0'*64,
                'models': {
                    MODEL_KEY: {
                        'status': 'failed',
                        'appearance': 'pbr_texture',
                        'error': error_msg,
                        'elapsed_seconds': elapsed,
                    },
                },
            }
            worker_out_path.write_text(json.dumps(failed_worker, indent=2))

        append_progress({
            'dish': label,
            'seconds': elapsed,
            'status': dish_status,
            'error': error_msg,
        })

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

finally:
    total_mins = round((time.perf_counter() - total_start_time) / 60.0, 2)
    print(f"\n=== Benchmark Finished: Total GPU time: {total_mins} minutes ===")
    if 'sys' in globals() and sys.exc_info()[0] is not None:
        print("Interrupted by exception. Emergency unmount and release...")
        from google.colab import drive
        drive.flush_and_unmount()
        if AUTO_RELEASE:
            print("Auto-releasing Colab runtime...")
            from google.colab import runtime
            runtime.unassign()


In [ ]:
# 7. Verification and Runtime Release
try:
    print("=== Verifying Colab Output Artifacts ===")
    completed = 0
    if OUTPUT_DIR.exists():
        for d in sorted(OUTPUT_DIR.iterdir()):
            if d.is_dir() and (d / 'worker_output.json').is_file():
                completed += 1
                rec = json.loads((d / 'worker_output.json').read_text())
                m_st = (rec.get('models') or {}).get(MODEL_KEY, {}).get('status')
                print(f"  Dish '{d.name}': status={m_st}")
    print(f"Total completed dish records: {completed}")
finally:
    print("Flushing Google Drive...")
    from google.colab import drive
    drive.flush_and_unmount()
    print("[OK] Google Drive flushed and unmounted.")
    if AUTO_RELEASE:
        print("AUTO_RELEASE is True: releasing Colab runtime now to avoid burning credits...")
        from google.colab import runtime
        runtime.unassign()
